# Component 9: Structured JSON Output

This notebook demonstrates how the metrics calculated across the entire pipeline are aggregated and serialized into the strict JSON schema required for BioGPT grounding.

In [ ]:
import sys
import json
import tempfile
from pathlib import Path

# Ensure our src imports work
sys.path.append('..')

from src.components.component9_json_output import generate_structured_json, save_structured_json

## 1. Aggregate Pipeline Metrics
Here we simulate collecting the various outputs from Component 1 (Encoder), Component 3 (Routing), Component 4 (Lesion Mask), Component 7 (Boundary Critic), and Component 8 (Timika Score).

In [ ]:
pipeline_results = {
    "patient_id": "SHZ_0042",
    "modality": "CXR-PA",
    "scanner_domain": "US-CR",
    "n_distinct_lesions": 3,
    "lesion_area_cm2": 18.529,  # Will be mathematically rounded to 18.53
    "expert_routing": {"E1": 0.82, "E2": 0.12, "E3": 0.05, "E4": 0.01},
    "boundary_quality_score": 0.912,
    "fp_probability": 0.041,
    "alp": 42.68,               # Will be mathematically rounded to 42.7
    "cavity_flag": 1,
    "timika_score": 82.68,      # Will be mathematically rounded to 82.7
    "severity": "moderate",
    "pathology_flags": {
        "consolidation": True,
        "cavitation": True,
        "effusion": False,
        "fibrosis": False
    }
}

## 2. Generate and Validate JSON
We pass the results into Component 9 to enforce constraints (like `cavitation_confidence: "radiographic-only"`) and precision formatting.

In [ ]:
# Generate the strict dictionary
structured_output = generate_structured_json(**pipeline_results)

# Display the formatted JSON (this is exactly what goes to BioGPT)
print(json.dumps(structured_output, indent=2))

# Verify specific constraints
assert structured_output['scoring']['cavitation_confidence'] == "radiographic-only", "Missing forced confidence clause!"
assert structured_output['scoring']['ALP'] == 42.7, "Failed precision rounding limit!"
print("\n\u2705 Schema validation passed!")

## 3. Save Artifact
This persists the file to disk so BioGPT can load it during inference.

In [ ]:
# Save it to a temporary test file
temp_dir = tempfile.gettempdir()
target_path = Path(temp_dir) / "pipeline_SHZ_0042.json"

save_structured_json(structured_output, str(target_path))

print(f"Correctly persisted artifact to: {target_path}")